# Cognopolis · Урок M6 — староста «город умов»

До сих пор ты управлял одним жителем. Теперь у аккаунта **несколько жителей** (1 житель = 1 агент), у поселения — **доска целей**, а между ними пусто: **сервер не распределяет работу**. Этот зазор закрывает **староста** — житель, чей под-токен получил право писать поручения всему ростеру. Его мозг — код этого урока.

**Что построим.** Шляпой игрока — фундамент эпохи старосты (склад ур.2 → Ратуша ур.3 → найм Кьюби и Тихона), доску целей и роль старосты. Затем — мозг старосты как **LangGraph-граф** `observe → plan → assign` (обычные python-функции над typed-состоянием, **без LLM**) и «час работников»: каждый житель своим под-токеном исполняет своё поручение реактивной петлёй из M1.

**Тир агента:** мультиагент (майлстоун игры **M6**). **Дальше по курсу:** оценивание агентов (M7).

> ⚙️ **Working-first.** Ноутбук рассчитан на прогон `Run all` без правок — нужны лишь адрес мира и **логин + пароль** аккаунта (`COGNOPOLIS_USERNAME` / `COGNOPOLIS_PASSWORD`): найм и назначение старосты — действия игрока, им нужна сессия. Учебная активность — в секции **«Задачи»**. LLM-ключ (MiniMax) нужен только задаче 1 — базовый путь бегает keyless.

**Ссылки** (подставь адрес своего мира вместо `<BASE_URL>`):
- API-доки (Swagger, кнопка **Authorize**): `<BASE_URL>/docs`
- 👀 Смотреть за жителями в браузере: `<BASE_URL>/?token=<под-токен жителя>` (read-only, по вкладке на жителя)
- Контракт: агент видит мир **только** через API (`cognopolis_client`).
- Раньше этого — пройди **планировщик (M3)**: оттуда экономика склада, стройка и поручения. Уроки M4–M5 в разработке — для M6 они не нужны.


## 1. Сетап

Ставим официальный клиент игры из git-URL (PyPI пока нет) и **LangGraph** — на нём соберём мозг старосты.


In [ ]:
%pip install -q "cognopolis-client @ git+https://github.com/ITrubnikov/Train_of_Thought-Cognopolis.git#subdirectory=client" langgraph


In [ ]:
import os
from cognopolis_client import Client, GameError

# ⬇️ ЖИВОЙ МИР COGNOPOLIS (публичный инстанс). Можно переопределить переменной COGNOPOLIS_URL.
BASE_URL = os.environ.get("COGNOPOLIS_URL", "https://kindomklaster.com")

def get_secret(name: str) -> str:
    """Секрет из env / Colab / Kaggle — одним хелпером (канон курса)."""
    val = os.environ.get(name, "")
    if val:
        return val
    try:
        from google.colab import userdata            # Colab: значок ключа слева -> Secrets
        return userdata.get(name) or ""
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle: Add-ons -> Secrets
        return UserSecretsClient().get_secret(name) or ""
    except Exception:
        pass
    return ""

# ⬇️ ЛОГИН И ПАРОЛЬ АККАУНТА (не только токен!): найм и назначение старосты — действия игрока.
#   Используй аккаунт из M1–M3: на проде регистрация закрыта, вход — только логином.
USERNAME = get_secret("COGNOPOLIS_USERNAME") or ""   # ← или впиши логин в кавычки
PASSWORD = get_secret("COGNOPOLIS_PASSWORD") or ""   # ← или впиши пароль в кавычки
assert USERNAME and PASSWORD, (
    f"Нужны логин и пароль аккаунта {BASE_URL} — секреты COGNOPOLIS_USERNAME / COGNOPOLIS_PASSWORD "
    "(или впиши их в эту ячейку).")

c = Client(BASE_URL)

# Мягкая проверка связи: если мир недоступен — не пугаем трейсбеком, живые ячейки ниже пропустятся.
WORLD_UP = True
try:
    Client(BASE_URL).get_map()  # GET /map не требует токена
except Exception as e:
    WORLD_UP = False
    print(f"⚠️  Мир {BASE_URL} недоступен ({type(e).__name__}). Живые ячейки пропущу — "
          "проверь COGNOPOLIS_URL или попробуй позже.")

if WORLD_UP:
    try:
        c.login(USERNAME, PASSWORD)          # сессия игрока + под-токен ПЕРВОГО жителя в c.token
        print("Вошли как", USERNAME)
    except GameError:
        try:
            c.register(USERNAME, PASSWORD)   # локальный мир: регистрация открыта
            print("Создали аккаунт", USERNAME)
        except GameError as e:
            raise RuntimeError(
                f"Не удалось войти ({e.code}). На проде регистрация закрыта — "
                "используй логин/пароль существующего аккаунта из M1.") from None

print("Мир:", BASE_URL, "| на связи:", WORLD_UP)


## 2. Разогрев — шляпа игрока: фундамент эпохи старосты

Староста появляется не на пустом месте. Прежде чем агент-координатор обретёт смысл, игроку нужно построить экономический фундамент (всё платится со **склада**):

| Ступень | Стоимость | Зачем |
| --- | --- | --- |
| склад ур.2 | 12 дерева + 8 камня | вместимость 50 → 100: иначе бюджет найма не поместится |
| Ратуша ур.2 | 15 дерева + 10 камня | лимит ростера = уровень Ратуши |
| Ратуша ур.3 | 30 дерева + 20 камня | три жителя + **право писать доску целей** |
| найм Кьюби | 20 дерева + 10 камня | вторые руки |
| найм Тихона | 40 дерева + 20 камня | третьи руки (найм дорожает с размером ростера) |

Гриндим это компактной **реактивной петлёй из M1** — ничего нового: к ноде → добыть → с полным рюкзаком домой → авто-банк. С чистого аккаунта — несколько минут; после M3 быстрее: топор со склада бустит добычу любому жителю, а Ратуша ур.2 уже стоит. Каждая ступень идемпотентна — повторный `Run all` пропустит готовое.


In [ ]:
import time

HOME = (0, 0)                                      # дом: шаг сюда авто-банкает рюкзак на склад
RESOURCE_NODE = {"wood": "tree", "stone": "rock"}  # сырьё -> клетка-нода на карте
RU = {"wood": "дерево", "stone": "камень", "town_hall": "Ратуша", "storehouse": "склад",
      "sawmill": "лесопилка", "forge": "кузница", "goblin": "гоблин", "wolf": "волк"}

def carried_total(ch):
    return sum(ch["inventory"].values())

def levels(ch):
    """Уровни зданий базы (Ратуша и склад есть всегда, ур.1 по умолчанию)."""
    return {b["kind"]: b["level"] for b in ch["buildings"]}

def nearest(ch, tiles, content):
    """Ближайшая клетка с заданным содержимым (по манхэттенскому расстоянию)."""
    nodes = [t for t in tiles if t["content"] == content]
    return min(nodes, key=lambda t: abs(t["x"] - ch["x"]) + abs(t["y"] - ch["y"]))

def adjacent(ch, tx, ty):
    """Сосед или та же клетка (действовать можно вплотную)."""
    return abs(ch["x"] - tx) + abs(ch["y"] - ty) <= 1

def step_toward(ch, tx, ty):
    """Один шаг к цели: сперва по X, потом по Y (карта без стен)."""
    if ch["x"] != tx:
        return ch["x"] + (1 if tx > ch["x"] else -1), ch["y"]
    return ch["x"], ch["y"] + (1 if ty > ch["y"] else -1)

def gather_tick(cl, world, res, target, reason):
    """Один виток «добывать res, пока на складе не будет target». Возвращает "done" | "работаю"."""
    ch = cl.get_character()                                        # observe
    if ch["stored"].get(res, 0) >= target:
        return "done"                                              # цель по складу закрыта
    enough = ch["stored"].get(res, 0) + ch["inventory"].get(res, 0) >= target
    try:
        if (enough or carried_total(ch) >= ch["inventory_cap"]) and (ch["x"], ch["y"]) != HOME:
            r = cl.move(*step_toward(ch, *HOME), reason=reason)    # несём добычу на склад
            nch = r["character"]
            if (nch["x"], nch["y"]) == HOME and carried_total(nch) > 0 and not r["result"].get("banked"):
                print("⚠️ Склад полон — улучши склад: c.upgrade('storehouse')")
        else:
            node = nearest(ch, world["tiles"], RESOURCE_NODE[res])
            if adjacent(ch, node["x"], node["y"]):
                cl.gather(res, reason=reason)                      # у ноды — добыча (ресурс явно)
            else:
                cl.move(*step_toward(ch, node["x"], node["y"]), reason=reason)
    except GameError as e:
        print("  шаг не прошёл:", e.code)
        time.sleep(1.0)                    # после ошибки кулдаун неизвестен — не лупим сервер
    cl.wait_cooldown()                     # wait: ритм диктует сервер
    return "работаю"

def ensure_stored(cl, world, cost, why):
    """Шляпа игрока: догриндить склад до cost. Возвращает потраченные тики."""
    ticks = 0
    for res, qty in cost.items():
        for _ in range(400):
            if gather_tick(cl, world, res, qty, reason=f"{why}: нужно {qty} {RU[res]}") == "done":
                break
            ticks += 1
    return ticks


In [ ]:
GRIND_TICKS = 0

if WORLD_UP:
    world = c.get_map()                   # ноды сырья статичны — карту берём один раз

    def stage(label, cost, ready, act):
        """Одна ступень фундамента: если ещё не готово — догриндить склад и выполнить."""
        global GRIND_TICKS
        if ready():
            print(f"· {label} — уже есть, пропускаю")
            return
        GRIND_TICKS += ensure_stored(c, world, cost, label)
        act()
        c.wait_cooldown()
        print(f"· {label} — готово")

    th   = lambda: levels(c.get_character()).get("town_hall", 1)
    sh   = lambda: levels(c.get_character()).get("storehouse", 1)
    crew = lambda: len(c.get_roster()["characters"])

    stage("склад ур.2 (вместимость 100)", {"wood": 12, "stone": 8},
          lambda: sh() >= 2, lambda: c.upgrade("storehouse", reason="фундамент эпохи старосты"))
    stage("Ратуша ур.2", {"wood": 15, "stone": 10},
          lambda: th() >= 2, lambda: c.upgrade("town_hall", reason="лимит ростера = уровень Ратуши"))
    stage("Ратуша ур.3 (доска целей + три жителя)", {"wood": 30, "stone": 20},
          lambda: th() >= 3, lambda: c.upgrade("town_hall", reason="эпоха старосты"))
    stage("найм: Кьюби", {"wood": 20, "stone": 10},
          lambda: crew() >= 2, lambda: c.hire("Кьюби", reason="в поселении нужны руки"))
    stage("найм: Тихон", {"wood": 40, "stone": 20},
          lambda: crew() >= 3, lambda: c.hire("Тихон", reason="третьи руки — под третью цель"))

    print("Грайнд-тиков потрачено:", GRIND_TICKS,
          "| ростер:", [r["name"] for r in c.get_roster()["characters"]])


### Доска целей и роль старосты

Фундамент готов — можно писать доску целей: до Ратуши ур.3 эта запись падала бы с `town_hall_locked` (мир выдаёт доску тому, у кого есть кому её раздавать; читать её можно всегда). Доска пишется под-токеном, а вот **роль старосты назначает игрок сессией** — как и найм. Под-токены всех жителей соберём повторным `login()`: в ростере токенов не бывает, они приезжают только в ответах `login()` и `hire()`.


In [ ]:
if WORLD_UP:
    ch = c.get_character()
    WOOD_GOAL  = ch["stored"].get("wood", 0) + 10    # цели считаем ОТ текущего склада,
    STONE_GOAL = ch["stored"].get("stone", 0) + 6    # чтобы повторный Run all тоже работал

    board = c.set_settlement_goals([                 # гейт: Ратуша ур.3 (иначе town_hall_locked)
        {"type": "gather", "resource": "wood",  "target": WOOD_GOAL,
         "flavor": "к зиме нужен запас дров"},
        {"type": "gather", "resource": "stone", "target": STONE_GOAL,
         "flavor": "камень на фундамент кузницы"},
        {"type": "defeat", "enemy": "goblin",
         "flavor": "гоблин пугает сборщиков у леса — сначала разберитесь!"},
    ])
    print(f"Доска целей (версия {board['version']}):")
    for g in board["goals"]:
        print("  -", {k: v for k, v in g.items() if v is not None})

    # Под-токены ВСЕХ жителей: повторный login() возвращает весь ростер с токенами.
    data = c.login(USERNAME, PASSWORD)
    TOKENS = {r["id"]: r["token"] for r in data["characters"]}
    NAMES  = {r["id"]: r["name"]  for r in data["characters"]}
    ELDER_ID = data["characters"][0]["id"]           # старостой назначим стартового жителя

    c.appoint_elder(ELDER_ID)                        # действие игрока (сессия); None — снять роль
    elder = Client(BASE_URL, token=TOKENS[ELDER_ID]) # мозг старосты ходит ЕГО под-токеном

    print()
    for r in c.get_roster()["characters"]:
        skills = {k: v["level"] for k, v in r["skills"].items()}
        print(f"{r['name']:>8}: role={r['role']:<6} hp={r['hp']:>2} tools={r['tools']} skills={skills}")


## 3. Разбор — мозг старосты как граф

Между доской целей и слотами поручений сервер оставил пустоту — её закрывает агент M6. Собираем его на **LangGraph**: мозг старосты = граф из узлов-этапов над типизированным состоянием. LLM в базовом пути нет — узлы это обычные python-функции:

- **observe** — только читает мир: доска + ростер + карта. Ни решений, ни записей.
- **plan** — чистая функция матчинга: цели доски × способности жителей (`skills` / `tools` / `hp`) → `{житель: поручение}`. Её можно тестировать без сервера.
- **assign** — единственный узел с побочными эффектами: пишет слоты под-токеном старосты (кросс-запись — право роли `elder`; в Хронике жителя останется «староста поручил: …»).
- **report** — ветка «не хватило рук»: нераспределённые цели — это выход графа, а не потерянный лог. Сюда ведёт **условное ребро** после `assign`.

Матчинг v1 — детерминированный: дерево — лучшему `woodcutting` (топор в плюс), камень — лучшему `mining`, в бой — самый здоровый, стройка — любому свободному. Два правила повторного витка: слот занят, но его цель уже закрыта → житель считается **свободным** (это даст перебалансировку бесплатно, задача 2); цель уже висит в чьём-то живом слоте → второй раз **не раздаётся** (иначе двое побегут за одним гоблином). Перезапись слота — last-write-wins, но с подписью актёра в Хронике.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class ElderState(TypedDict, total=False):
    goals: list         # доска целей (в порядке доски)
    roster: list        # жители: способности + текущие поручения
    enemies: list       # живность на карте (для целей defeat)
    assignments: dict   # решение: character_id -> спека поручения
    unassigned: list    # цели, на которые не хватило рук

GATHER_SKILL = {"wood": "woodcutting", "stone": "mining"}
GATHER_TOOL  = {"wood": "axe",         "stone": "pickaxe"}

def goal_done(goal, state):
    """Закрыта ли цель — по общему складу / зданиям / карте. Чистая функция от состояния."""
    base = state["roster"][0]                       # склад и здания общие — видны любому жителю
    if goal["type"] == "gather":
        return base["stored"].get(goal["resource"], 0) >= goal["target"]
    if goal["type"] == "build":
        return any(b["kind"] == goal["structure"] for b in base["buildings"])
    if goal["type"] == "defeat":
        return not any(e["kind"] == goal["enemy"] and e["alive"] for e in state["enemies"])
    return False

def slot_free(resident, state):
    """Жителя можно нагружать: слот пуст ИЛИ его прежняя цель уже закрыта (перебалансировка)."""
    a = resident["assignment"]
    return a is None or goal_done(a, state)

def task_key(spec):
    return (spec.get("type"), spec.get("resource"), spec.get("target"),
            spec.get("enemy"), spec.get("structure"))

def covered(goal, state):
    """Цель уже у кого-то в работе (слот совпадает и ещё не закрыт)? Не раздаём дважды."""
    return any(r["assignment"] and task_key(r["assignment"]) == task_key(goal)
               and not goal_done(r["assignment"], state) for r in state["roster"])

def ability(resident, goal):
    """Скор «насколько житель подходит цели» — детерминированный (v1, без LLM)."""
    if goal["type"] == "gather":
        xp = resident["skills"].get(GATHER_SKILL[goal["resource"]], {}).get("xp", 0)
        return xp + (1000 if GATHER_TOOL[goal["resource"]] in resident["tools"] else 0)
    if goal["type"] == "defeat":
        return resident["hp"]                       # в бой — самый здоровый
    return 1                                        # build: любой свободный

def short(spec):
    return f"{spec['type']} {spec.get('resource') or spec.get('enemy') or spec.get('structure') or ''}".strip()

# --- Узлы графа: обычные python-функции state -> частичное обновление state ---

def observe(state):
    """Только читает мир. Никаких решений и записей."""
    goals   = elder.get_settlement_goals()["goals"]
    roster  = elder.get_roster()["characters"]
    enemies = elder.get_map()["enemies"]
    print(f"· observe: целей на доске {len(goals)}, жителей {len(roster)}")
    return {"goals": goals, "roster": roster, "enemies": enemies}

def plan(state):
    """Чистый матчинг: жадно раздаём открытые цели свободным жителям по способностям."""
    free = {r["id"]: r for r in state["roster"] if slot_free(r, state)}
    assignments, unassigned = {}, []
    for goal in state["goals"]:
        if goal_done(goal, state):
            print("· plan: цель уже закрыта —", goal.get("flavor") or short(goal))
            continue
        if covered(goal, state):
            print("· plan: цель уже в работе —", goal.get("flavor") or short(goal))
            continue
        if not free:
            unassigned.append(goal)
            continue
        best = max(free.values(), key=lambda r: ability(r, goal))
        del free[best["id"]]
        assignments[best["id"]] = {k: v for k, v in goal.items() if v is not None}
    return {"assignments": assignments, "unassigned": unassigned}

def assign(state):
    """Единственный узел с побочными эффектами: пишем слоты под-токеном СТАРОСТЫ."""
    for cid, spec in state["assignments"].items():
        elder.set_assignment(cid, spec)             # чужие слоты — право роли elder
        print(f"· assign: {NAMES.get(cid, cid)} <- {short(spec)}")
    return {}

def report(state):
    """Ветка «не хватило рук»: нераспределённое — выход графа, а не потерянный лог."""
    for goal in state["unassigned"]:
        print("· report: некому поручить —", goal.get("flavor") or short(goal))
    return {}

g = StateGraph(ElderState)
g.add_node("observe", observe)
g.add_node("plan", plan)
g.add_node("assign", assign)
g.add_node("report", report)
g.set_entry_point("observe")
g.add_edge("observe", "plan")
g.add_edge("plan", "assign")
g.add_conditional_edges("assign", lambda s: "report" if s["unassigned"] else END,
                        {"report": "report", END: END})  # все цели розданы? -> END | report
g.add_edge("report", END)
elder_app = g.compile()
print("Граф старосты собран: observe -> plan -> assign -> (report | END)")


In [ ]:
def run_elder():
    """Один виток старосты: прогнать граф и вернуть итоговое состояние."""
    return elder_app.invoke({"goals": [], "roster": [], "enemies": [],
                             "assignments": {}, "unassigned": []})

if WORLD_UP:
    final = run_elder()

    print("\nСлоты после витка старосты:")
    for r in c.get_roster()["characters"]:
        a = r["assignment"]
        print(f"{r['name']:>8}: {short(a) if a else 'слот пуст'}"
              + (f" | от: {a['assigned_by']}" if a else ""))


## 4. Час работников — pull и под-токены

Поручения лежат в слотах, но никто никого не «будил»: push-а в мире нет. Каждый житель **сам** узнаёт своё поручение на очередном витке — `get_assignment()` тем же pull-ом, что в M1. Это blackboard-координация: общее состояние вместо звонков друг другу; она переживает перезапуски и рассинхрон любых агентов. И помни: поручение — задача, не приказ. Мир не заставит жителя работать — просто интерфейс подсветит расхождение.

Ниже город шевелится: каждый житель работает **своим под-токеном** — это три отдельных агента, а не три потока одного жителя (кулдауны и лимиты у каждого свои). `work_tick` — один виток реактивной петли: сбор как в M1, бой с hp-гардом как в M2, стройка как в M3.


In [ ]:
def work_tick(wc, world, a):
    """Один виток жителя над СВОИМ поручением. Возвращает "done" | "работаю" | "свободен"."""
    if a is None:
        return "свободен"                     # слот пуст — житель волен заниматься своим
    if a["type"] == "gather":                 # сбор — та же петля, что гриндила фундамент
        return gather_tick(wc, world, a["resource"], a["target"],
                           reason=f"поручение старосты: {RU[a['resource']]} до {a['target']}")
    ch = wc.get_character()                   # observe
    try:
        if a["type"] == "defeat":
            enemy = next((e for e in wc.get_map()["enemies"]
                          if e["kind"] == a["enemy"] and e["alive"]), None)
            if enemy is None:
                return "done"                 # враг повержен (может, соседом) — до респауна
            if ch["hp"] < 15:
                wc.rest(reason="лечусь перед боем — поручение старосты")   # hp-гард из M2
            elif adjacent(ch, enemy["x"], enemy["y"]):
                r = wc.fight(reason=f"поручение старосты: победить ({RU[a['enemy']]})")
                if r["result"].get("outcome") == "win":
                    return "done"
            else:
                wc.move(*step_toward(ch, enemy["x"], enemy["y"]),
                        reason="иду к врагу — поручение старосты")
        elif a["type"] == "build":
            if any(b["kind"] == a["structure"] for b in ch["buildings"]):
                return "done"
            wc.build(a["structure"], reason="поручение старосты")  # платит со склада
    except GameError as e:
        print("   шаг не прошёл:", e.code)
        time.sleep(1.0)
    wc.wait_cooldown()
    return "работаю"

if WORLD_UP:
    world = c.get_map()
    print("👀 Вкладки наблюдения (по одной на жителя):")
    for cid, name in NAMES.items():
        print(f"   {name}: {BASE_URL}/?token={TOKENS[cid]}")
    print()
    for cid, name in NAMES.items():
        wc = Client(BASE_URL, token=TOKENS[cid])   # СВОЙ под-токен — свой агент
        status = "работаю"
        for tick in range(40):
            a = wc.get_assignment()["assignment"]  # pull: житель узнаёт поручение на СВОЁМ витке
            status = work_tick(wc, world, a)
            if status in ("done", "свободен"):
                break
        print(f"— {name}: {status} (тиков: {tick + 1})")


## 5. Задачи

Каркас работает — три задачи (раскомментируй блок и прогони ячейку; заготовки рабочие):

1. **LLM-узел приоритета (MiniMax).** Игрок пишет в `flavor` целей живой текст. Вставь между `observe` и `plan` узел, который переставляет доску по срочности. Ключ — секрет `MINIMAX_API_KEY` (Kaggle/Colab/env); **без ключа узел деградирует в no-op** — ноутбук живёт keyless.
2. **Перебалансировка.** Освободи слот Кьюби, допиши на доску новую цель и прогони граф ещё раз: закрытые цели выпадут сами, а новое достанется **лучшим свободным рукам** — не обязательно тому, кого ты освободил. Replan из M3 — теперь на уровне поселения.
3. **Blast-radius руками.** Сними старосту — и убедись, что кросс-запись его токеном стала `404`. Верни роль — и право вернулось.


In [ ]:
# --- Задача 1. LLM-узел приоритета (MiniMax) ---------------------------------
# Детерминированный plan читает доску по порядку. LLM-узел переставит цели по срочности,
# прочитав flavor как живой текст (то, что в M3 было «lookup по каталогу слов»).
# Узел ЧЕСТНО деградирует в no-op без ключа — базовый путь остаётся keyless.

# import json, re
# import httpx
#
# MINIMAX_KEY = get_secret("MINIMAX_API_KEY")     # Kaggle/Colab-секрет или переменная окружения
#
# def prioritize(state):
#     """LLM-узел: доска -> порядок по срочности. Нет ключа/ответа -> no-op (порядок доски)."""
#     if not MINIMAX_KEY:
#         print("· prioritize: ключа MiniMax нет — оставляю порядок доски (no-op)")
#         return {}
#     numbered = "\n".join(f"{i}: {json.dumps(g, ensure_ascii=False)}"
#                          for i, g in enumerate(state["goals"]))
#     try:
#         resp = httpx.post("https://api.minimax.io/v1/chat/completions",
#             headers={"Authorization": f"Bearer {MINIMAX_KEY}"},
#             json={"model": "MiniMax-M3", "messages": [
#                 {"role": "system", "content":
#                  "Ты — староста поселения. Отсортируй цели по срочности, судя по их flavor. "
#                  "Ответь ТОЛЬКО JSON-списком индексов, например [2, 0, 1]."},
#                 {"role": "user", "content": numbered}]},
#             timeout=60)
#         content = resp.json()["choices"][0]["message"]["content"]
#         order = json.loads(re.findall(r"\[[\d,\s]*\]", content)[-1])
#         assert sorted(order) == list(range(len(state["goals"])))
#         print("· prioritize: порядок от LLM:", order)
#         return {"goals": [state["goals"][i] for i in order]}
#     except Exception as e:
#         print(f"· prioritize: LLM не ответила ({type(e).__name__}) — no-op")
#         return {}
#
# g2 = StateGraph(ElderState)
# g2.add_node("observe", observe)
# g2.add_node("prioritize", prioritize)   # новый узел встал МЕЖДУ observe и plan —
# g2.add_node("plan", plan)               # остальные узлы не тронуты вообще
# g2.add_node("assign", assign)
# g2.add_node("report", report)
# g2.set_entry_point("observe")
# g2.add_edge("observe", "prioritize")
# g2.add_edge("prioritize", "plan")
# g2.add_edge("plan", "assign")
# g2.add_conditional_edges("assign", lambda s: "report" if s["unassigned"] else END,
#                          {"report": "report", END: END})
# g2.add_edge("report", END)
# elder_app = g2.compile()                # run_elder() теперь ходит через новый граф
# if WORLD_UP:
#     run_elder()


In [ ]:
# --- Задача 2. Перебалансировка ----------------------------------------------
# Слоты не самоочищаются: выполненное поручение висит, пока его не перепишут. Наш plan
# считает жителя свободным, если слот пуст ИЛИ его цель уже закрыта (slot_free), — значит,
# новый виток графа сам перераздаст открытое. Освободим Кьюби, допишем на доску новую цель
# и посмотрим: её получат ЛУЧШИЕ свободные руки (не обязательно Кьюби — матчинг честный).

# if WORLD_UP:
#     kyubi = next(cid for cid, n in NAMES.items() if n == "Кьюби")
#     elder.clear_assignment(kyubi)              # слот пуст — «жителя потеряли»
#     goals = c.get_settlement_goals()["goals"]  # игрок дописал цель на доску:
#     goals.append({"type": "gather", "resource": "stone",
#                   "target": STONE_GOAL + 8, "flavor": "камня много не бывает"})
#     c.set_settlement_goals(goals)
#     run_elder()                                # новый виток: закрытое выпало, новое разошлось
#     for r in c.get_roster()["characters"]:
#         a = r["assignment"]
#         print(f"{r['name']:>8}:", short(a) if a else "слот пуст")
#     for cid, name in NAMES.items():            # мини-час работников: разобрать новое
#         wc = Client(BASE_URL, token=TOKENS[cid])
#         status = "работаю"
#         for tick in range(40):
#             status = work_tick(wc, world, wc.get_assignment()["assignment"])
#             if status in ("done", "свободен"):
#                 break
#         print(f"— {name}: {status}")

# --- Задача 3. Blast-radius руками --------------------------------------------
# Снимем роль — и убедимся, что кросс-запись сжалась до 404 в ту же секунду:

# if WORLD_UP:
#     c.appoint_elder(None)                      # роль снята (действие игрока, сессия)
#     target = next(cid for cid in TOKENS if cid != ELDER_ID)
#     try:
#         elder.set_assignment(target, {"type": "gather", "resource": "wood", "target": 5})
#         raise AssertionError("кросс-запись прошла БЕЗ роли старосты — так быть не должно")
#     except GameError as e:
#         print("кросс-запись без роли:", e.status_code, e.code)  # 404 — мир не признаётся, есть ли слот
#         assert e.status_code == 404
#     c.appoint_elder(ELDER_ID)                  # вернём роль — Run all остаётся повторяемым
#     elder.clear_assignment(target)             # право вернулось: кросс-запись снова работает
#     run_elder()                                # и сразу перераздать освободившийся слот


## 6. Проверка

Критерий приёма: поручения розданы **по способностям**, в Хронике рабочих видна рука старосты («староста поручил: …»), а склад дорос до целей с доски. Если ассерты падают — просто перезапусти ячейки `run_elder()` и «час работников»: и граф, и петля жителей — функции от состояния, они продолжат с места остановки.


In [ ]:
if WORLD_UP:
    roster = c.get_roster()["characters"]
    stored = roster[0]["stored"]                     # склад общий

    # 1) поручения розданы (слоты живут, пока их не перепишут)
    assigned = [r for r in roster if r["assignment"]]
    assert len(assigned) >= 2, "Поручений меньше двух — прогони ячейку run_elder() ещё раз."

    # 2) матчинг по способностям: цель «дерево» — у лучшего лесоруба ростера
    wood_goal = {"type": "gather", "resource": "wood", "target": WOOD_GOAL}
    lumber = next((r for r in roster
                   if r["assignment"] and r["assignment"]["type"] == "gather"
                   and r["assignment"]["resource"] == "wood"), None)
    if lumber is not None:
        best = max(ability(r, wood_goal) for r in roster)
        assert ability(lumber, wood_goal) == best, \
            "Дерево поручено не лучшему лесорубу — проверь plan()/ability()."

    # 3) кросс-запись видна в Хронике: у кого-то из РАБОЧИХ есть «староста поручил»
    found = any(
        any(e["action"] == "assignment" and "староста поручил" in e["summary"]
            for e in Client(BASE_URL, token=TOKENS[cid]).get_events(limit=200))
        for cid in TOKENS if cid != ELDER_ID)
    assert found, "В Хронике рабочих нет «староста поручил» — поручения писал не токен старосты?"

    # 4) прогресс по целям: склад дорос до целей с доски
    assert stored.get("wood", 0) >= WOOD_GOAL, \
        "Дерево не добрано — дай работникам ещё тиков («час работников»)."
    assert stored.get("stone", 0) >= STONE_GOAL, \
        "Камень не добран — дай работникам ещё тиков («час работников»)."

    print("✅ Город умов работает: цели розданы по способностям, склад дорос до целей,",
          "в Хронике видна рука старосты.")
else:
    print("Мир недоступен — проверка пропущена.")


## Наблюдаемость — смотри на город, а не на жителя

Открой несколько вкладок `BASE_URL/?token=<под-токен>` — по одной на жителя — и прогони «час работников» ещё раз:

- у каждого жителя свой мысль-пузырь с `reason` («поручение старосты: …») — видно, что работают три разных агента;
- в **Ратуше → «поручения»** — слоты всех жителей рядом с доской целей: раздача старосты как на ладони;
- в **Хронике** каждого рабочего — запись «староста поручил: …»: перезапись слота всегда подписана актёром (last-write-wins с подписью);
- в **«Жителях»** — строка «Поручение vs Сейчас» и роль `elder` у старосты.

**Что дальше.** Город работает — но насколько хорошо? Следующий урок — **оценивание агентов (M7)**: метрики и сравнение стратегий старосты. А когда выйдут M4–M5 (LLM tool-use и торговля), их место в этой архитектуре уже готово: LLM — это узел графа (ты сделал его в задаче 1), а не новый агент с нуля.
